# FrameWeave Ovis2.5-9B ablations

Colab training notebook (**A100 40 GB**). Sáu biến thể dùng chung schedule, loss và backbone: `vanilla`, `token_mlp`, `global_only`, `local_only`, `hybrid`, `hybrid_no_time`. `hybrid` là `RelJoinResidual(variant="Hybrid")` của V6. Notebook này chỉ gọi `run_ablation.py` trong `/content/ovis_env`.

$$
\widetilde x_{ip} = x_{ip} + m_{ip}\,\chi_i\,a_{ip}\,c_{ip}\,u_{ip}
$$

- Backbone [`ATH-MaaS/Ovis2.5-9B`](https://huggingface.co/ATH-MaaS/Ovis2.5-9B) đóng băng, revision `5a1f9b995c98ae5b85a28ec89d88d74295be0ea7`. Adapter fp32. Thinking **off**. 448×448.
- Loss V6: $\mathrm{logsumexp}(\mathcal L_{\mathrm{cand}},\mathcal L_{\mathrm{corr}})-\log 2$. Chọn epoch theo validation $L$, patience 3, tối đa 8 epoch.
- Split do `prepare` dựng bằng đúng hàm V6 sau khi notebook tải PerceptionTest: 454 câu / 128 video train, 67 câu / 19 video val. Không upload manifest.
- Torch trong `/content/ovis_env` (Python 3.11 + torch 2.4.0 + flash-attn 2.7). Không DTR. Không nâng package.

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `ovis/reljoin`, `ablation/`, `configs/` và `run_ablation.py` vào `/content`. Không upload file source.

Không upload `train_manifest.json` hay `val_manifest.json`. Notebook tải PerceptionTest, `prepare` ghi split vào output, cùng kiểu V6. Model vào `/content/checkpoints/ovis2_5_9b`, video vào `/content/perceptiontest_lite`. HERBench là notebook eval. Output: `/content/outputs/frameweave_ovis_ablation/`. Drive: `/content/drive/MyDrive/frameweave_ovis_ablation_outputs`.

## `RUN_MODE`

`unit` | `smoke` | `train`

`unit` không tải model hay video. `smoke` là hai update thật, không phải số paper. `train` chạy suite và resume được từ Drive.


## 1. Configuration

**Input:** none · **Output:** `RUN_MODE`, pins, paths


In [ ]:
# 1. Configuration — change RUN_MODE / VARIANT here
from __future__ import annotations
from pathlib import Path

RUN_MODE = "unit"  # unit | smoke | train
VARIANT = "all"  # all | vanilla | token_mlp | global_only | local_only | hybrid | hybrid_no_time
RESUME = False
SEED = 42

ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

MODEL_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
TRAIN_DATASET_ID = "lmms-lab/LLaVA-Video-178K"
TRAIN_DATASET_REVISION = "6d8c562dc26d70042a0d9704d1cae58c94b89098"

VENV = ROOT / "ovis_env"
PYTHON = VENV / "bin" / "python"
ABLATION = ROOT
V6 = ROOT
OUT_DIR = ABLATION / "outputs" / "frameweave_ovis_ablation"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_ablation_outputs")

print({
    "RUN_MODE": RUN_MODE,
    "VARIANT": VARIANT,
    "RESUME": RESUME,
    "SEED": SEED,
    "ROOT": str(ROOT),
    "OUT_DIR": str(OUT_DIR),
    "backbone": MODEL_ID,
    "method": "FrameWeave-Ovis-Ablation",
})


## 2. Environment installation (`ovis_env`)

Official Ovis2.5-9B pins: `torch==2.4.0`, `transformers==4.51.3`, `flash-attn==2.7.0.post2`.
Stop if flash-attn install fails — the ViT hard-imports it.


In [ ]:
# 2. Bootstrap isolated ovis_env (Python 3.11 + torch 2.4.0 + flash-attn 2.7)
from __future__ import annotations
import os, subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "ovis_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs" / "frameweave_ovis_ablation" / "logs" / "cell4_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc

sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "datasets", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging",
])
print("Installing flash-attn==2.7.0.post2 (required by Ovis ViT)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, flash_attn; "
    "print(torch.__version__); print(transformers.__version__); print(flash_attn.__version__); "
    "print(torch.__file__); print(transformers.__file__)"
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/ovis_env/" in line, f"Package leaked outside venv: {line}"
print("ovis_env ready:", PYTHON)
print("LOG:", LOG)


In [ ]:
# 2b. Extra pip (same packages as the V6 notebook). Writes a log so a failure is inspectable.
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
LOG_DIR = ROOT / "outputs" / "frameweave_ovis_ablation" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / "cell5_pip.log"

if not PYTHON.exists():
    msg = (
        f"Missing {PYTHON}. Run the ovis_env bootstrap cell first "
        "(section 2), then re-run this cell.\n"
    )
    LOG.write_text(msg, encoding="utf-8")
    print(msg)
    raise FileNotFoundError(msg)

cmd = [
    str(PYTHON), "-m", "pip", "install",
    "--disable-pip-version-check",
    "--upgrade-strategy", "only-if-needed",
    "scikit-learn==1.3.2",
    "matplotlib==3.8.4",
    "numpy==1.25.0",
]
print("$", " ".join(cmd))
proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_text = f"$ {' '.join(cmd)}\nexit={proc.returncode}\n\n{proc.stdout or ''}"
LOG.write_text(log_text, encoding="utf-8")
print(proc.stdout or "")
print("LOG:", LOG)

persist = Path("/content/drive/MyDrive/frameweave_ovis_ablation_outputs")
if persist.parent.exists():
    try:
        persist.mkdir(parents=True, exist_ok=True)
        dest = persist / "logs" / LOG.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(log_text, encoding="utf-8")
        print("Copied log →", dest)
    except OSError as exc:
        print("Persist copy skipped:", exc)

if proc.returncode != 0:
    raise RuntimeError(f"cell 5 pip failed ({proc.returncode}). See {LOG}")
print("cell 5 pip OK")


## 3. Reproducibility — paths, Drive


In [ ]:
# 3. Paths, Drive. Resume copies the ablation tree back before a new train.
from __future__ import annotations
import os
import shutil
from pathlib import Path

ROOT = Path("/content")
ABLATION = ROOT
OUT_DIR = ABLATION / "outputs" / "frameweave_ovis_ablation"
MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PERSIST = Path("/content/drive/MyDrive/frameweave_ovis_ablation_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    PERSIST = OUT_DIR
PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)
os.environ["OVIS_ABLATION_ROOT"] = str(ROOT)

try:
    RESUME
except NameError:
    RESUME = False
if RESUME and PERSIST.resolve() != OUT_DIR.resolve() and any(PERSIST.iterdir()):
    shutil.copytree(PERSIST, OUT_DIR, dirs_exist_ok=True)
    print("Restored", PERSIST, "→", OUT_DIR)

print({
    "RELJOIN_PERSIST": str(PERSIST),
    "OUT_DIR": str(OUT_DIR),
    "MODEL_DIR": str(MODEL_DIR),
    "RESUME": RESUME,
})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


link(REPO_DIR / "ovis" / "reljoin", ROOT / "reljoin")
link(REPO_DIR / "ovis_ablation" / "ablation", ROOT / "ablation")
link(REPO_DIR / "ovis_ablation" / "configs", ROOT / "configs")
link(REPO_DIR / "ovis_ablation" / "run_ablation.py", ROOT / "run_ablation.py")
print("FrameWeave source linked from", REPO_DIR / "ovis_ablation", "and", REPO_DIR / "ovis" / "reljoin")


In [ ]:
# 4. Verify FrameWeave sources. The 454/67 split is built later from PerceptionTest.
from __future__ import annotations
from pathlib import Path

ROOT = Path("/content")
required = [
    ROOT / "run_ablation.py",
    ROOT / "configs" / "ovis_ablations.yaml",
    ROOT / "ablation" / "modules.py",
    ROOT / "ablation" / "train.py",
    ROOT / "ablation" / "eval_herbench.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "train.py",
    ROOT / "reljoin" / "losses.py",
    ROOT / "reljoin" / "data_perception.py",
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("sources OK", len(required))


## 5. Dataset / model download (gated by `RUN_MODE`)

`unit` tải không gì. `smoke` và `train` tải Ovis vào `/content/checkpoints/ovis2_5_9b` và PerceptionTest vào `/content/perceptiontest_lite`. Cell sau đó `prepare` mới ghi split 454/67. Notebook train không tải HERBench.


In [ ]:
# 5. Conditional downloads (no HERBench in this training notebook)
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell 2 trước"
os.environ["PATH"] = f"{ROOT / 'ovis_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"

MODEL_DIR = ROOT / "checkpoints" / "ovis2_5_9b"
MODEL_HF_ID = "ATH-MaaS/Ovis2.5-9B"
MODEL_REVISION = "5a1f9b995c98ae5b85a28ec89d88d74295be0ea7"
need_data = RUN_MODE in {"smoke", "train"}
print({"need_model": need_data, "need_perception": need_data, "RUN_MODE": RUN_MODE})

if need_data:
    print("Downloading Ovis2.5-9B (~18 GB bf16)...")
    MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    py = (
        "import os\n"
        "os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'\n"
        "from huggingface_hub import snapshot_download\n"
        f"snapshot_download(repo_id='{MODEL_HF_ID}', revision='{MODEL_REVISION}', "
        f"local_dir=r'{MODEL_DIR}', ignore_patterns=['*.msgpack','*.h5','flax_model*','tf_model*'])\n"
        "print('model OK')\n"
    )
    subprocess.run([str(PYTHON), "-c", py], check=True)
    assert (MODEL_DIR / "config.json").exists(), f"Missing {MODEL_DIR / 'config.json'}"

    print("Downloading PerceptionTest train shard...")
    perception = (
        "import sys\n"
        "from pathlib import Path\n"
        f"sys.path.insert(0, {str(ROOT)!r})\n"
        "from reljoin.data_perception import download_perception_lite\n"
        "ann, videos = download_perception_lite(Path('/content/perceptiontest_lite'))\n"
        "print(ann)\n"
        "print(videos)\n"
    )
    subprocess.run([str(PYTHON), "-c", perception], check=True)

print("Download step done for RUN_MODE=", RUN_MODE)


## 6. Prepare shared schedule, then unit tests


In [ ]:
# 6. prepare + unit. Split must stay 454/128 and 67/19.
from __future__ import annotations
import json
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
ABLATION = ROOT
OUT = ABLATION / "outputs" / "frameweave_ovis_ablation"
assert PYTHON.exists(), "Chạy cell 2 trước"

for stage in ("prepare", "unit"):
    print("$", PYTHON, "run_ablation.py", stage, flush=True)
    subprocess.run([str(PYTHON), "run_ablation.py", stage], cwd=str(ABLATION), check=True)

meta_path = OUT / "shared" / "schedule_meta.json"
try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"
if RUN_MODE in {"smoke", "train"}:
    assert meta_path.is_file(), "prepare did not write the V6 split after PerceptionTest download"
if meta_path.is_file():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print({
        "train_questions": meta["train_questions"],
        "train_videos": meta["train_videos"],
        "val_questions": meta["val_questions"],
        "val_videos": meta["val_videos"],
        "schedule_sha256": meta["schedule_sha256"][:12],
        "frames_ready": meta["frames"]["rows_ready"],
        "frames_total": meta["frames"]["rows_total"],
    })
    assert meta["train_questions"] == 454 and meta["train_videos"] == 128
    assert meta["val_questions"] == 67 and meta["val_videos"] == 19
else:
    print("unit: split 454/67 is written when smoke/train has downloaded PerceptionTest")
report = json.loads((OUT / "unit" / "invariant_report.json").read_text(encoding="utf-8"))
assert report["ok"] is True
print("token_mlp", report["token_mlp"])
print("Unit invariants passed. Smoke accuracy is not a paper result.")


## 7. Smoke or train

Một process, một backbone. `smoke` chỉ hai update. `train` ghi `best/adapter.pt` theo validation $L$ thấp nhất.


In [ ]:
# 7. Train FrameWeave ablations (gated by RUN_MODE)
from __future__ import annotations
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "ovis_env" / "bin" / "python"
ABLATION = ROOT

try:
    RUN_MODE
except NameError:
    RUN_MODE = "unit"
try:
    VARIANT
except NameError:
    VARIANT = "all"
try:
    RESUME
except NameError:
    RESUME = False

if RUN_MODE == "unit":
    print("RUN_MODE=unit — skip smoke/train")
else:
    assert RUN_MODE in {"smoke", "train"}, RUN_MODE
    cmd = [str(PYTHON), "run_ablation.py", RUN_MODE, "--variant", VARIANT]
    if RUN_MODE == "train" and RESUME:
        cmd.append("--resume")
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(ABLATION), check=True)
    if RUN_MODE == "smoke":
        print("Smoke finished. paper_result is false. Do not put this accuracy in the paper table.")


## 8. Validation $L$ and train-loss curves


In [ ]:
# 8. Plot val.jsonl and train.jsonl. Empty until smoke/train has written them.
from __future__ import annotations
import json
from pathlib import Path

ROOT = Path("/content")
try:
    SEED
except NameError:
    SEED = 42
OUT = ROOT / "outputs" / "frameweave_ovis_ablation"
seed_dir = OUT / f"seed_{SEED}"
variants = sorted(path for path in seed_dir.glob("*") if path.is_dir()) if seed_dir.is_dir() else []
if not variants:
    print("No seed outputs yet. Run smoke or train first.")
else:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for variant_dir in variants:
        val_path = variant_dir / "val.jsonl"
        if val_path.is_file():
            rows = [json.loads(line) for line in val_path.read_text(encoding="utf-8").splitlines() if line.strip()]
            axes[0].plot([row["epoch"] for row in rows], [row["val_loss"] for row in rows], marker="o", label=variant_dir.name)
        train_path = variant_dir / "train.jsonl"
        if train_path.is_file():
            rows = [json.loads(line) for line in train_path.read_text(encoding="utf-8").splitlines() if line.strip()]
            updates = [row for row in rows if row.get("event") == "update"]
            axes[1].plot([row["global_step"] for row in updates], [row["loss"] for row in updates], label=variant_dir.name)
    axes[0].set_title("validation L")
    axes[0].set_xlabel("epoch")
    axes[1].set_title("train loss per update")
    axes[1].set_xlabel("global step")
    axes[0].legend(fontsize=8)
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    dest = OUT / "shared" / "loss_curves.png"
    dest.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(dest, dpi=120)
    print("Wrote", dest)
    plt.show()


## 9. Export artifacts


In [ ]:
# 9. Copy outputs to Drive persist dir + zip
from __future__ import annotations
import shutil
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path("/content")
OUT = ROOT / "outputs" / "frameweave_ovis_ablation"
persist = Path("/content/drive/MyDrive/frameweave_ovis_ablation_outputs")
if not persist.parent.exists():
    persist = OUT
persist.mkdir(parents=True, exist_ok=True)
if OUT.exists() and persist.resolve() != OUT.resolve():
    shutil.copytree(OUT, persist, dirs_exist_ok=True)
    print("Copied", OUT, "→", persist)

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
zip_path = ROOT / f"frameweave_ovis_ablation_{stamp}"
if OUT.exists():
    shutil.make_archive(str(zip_path), "zip", root_dir=OUT)
    print("Zip:", f"{zip_path}.zip")
print("Persist:", persist)
